# Advanced Python 3 (Modern Concepts)

## Q16. What is the Global Interpreter Lock (GIL)?
### Answer:
The GIL allows only one thread to execute Python bytecode at a time — meaning Python threads can’t run CPU-bound tasks truly in parallel.

- Works fine for I/O-bound tasks (like web requests or file I/O).
- Limits true parallelism for CPU-heavy computations.
- Tip: Use multiprocessing (separate processes) for CPU-bound tasks.

In [2]:
import threading

count = 0
def increment():
    global count
    for _ in range(1_000_000):
        count += 1  # GIL prevents true parallel execution

threads = [threading.Thread(target=increment) for _ in range(2)]
[t.start() for t in threads]
[t.join() for t in threads]
print(count)

1462950


## Q17. Difference between threading, multiprocessing, and asyncio?
| Type              | Best for                     | True Parallelism           | Example Use            |
| ----------------- | ---------------------------- | -------------------------- | ---------------------- |
| `threading`       | I/O-bound                    | ❌ No (GIL)                 | Web requests, file I/O |
| `multiprocessing` | CPU-bound                    | ✅ Yes                      | Data processing, ML    |
| `asyncio`         | I/O-bound (many connections) | ⚡ Concurrent, not parallel | Async web servers      |

In [7]:
# Example: asyncio concurrent tasks
import asyncio

async def task(name):
    print(f"{name} started")
    await asyncio.sleep(1)
    print(f"{name} finished")

async def main():
    await asyncio.gather(task("A"), task("B"))

try:
    asyncio.run(main())
except RuntimeError:
    print("asyncio.run() cannot be called from a running event loop")
else: 
    print("Nothing went wrong")
finally:
    print("The 'try except' is finished")

asyncio.run() cannot be called from a running event loop
The 'try except' is finished


C:\Users\rabbi\Anaconda3\lib\site-packages\ipykernel_launcher.py:15: RuntimeWarning: coroutine 'main' was never awaited
  from ipykernel import kernelapp as app


## Q18. What are Python type hints and why use them?
### Answer:
Type hints improve readability and catch bugs early with tools like mypy or IDEs. Helps with static analysis and autocompletion.

In [9]:
def greet(name: str, age: int) -> str:
    return f"{name} is {age} years old"

print(greet("Alice", 25))

Alice is 25 years old


## Q19. What are metaclasses?
### Answer:
Metaclasses control how classes are created — they’re like “classes of classes.”

In [10]:
class Meta(type):
    def __new__(cls, name, bases, dct):
        print(f"Creating class {name}")
        return super().__new__(cls, name, bases, dct)

class MyClass(metaclass=Meta):
    pass

# Output: Creating class MyClass

Creating class MyClass


### Use metaclasses for enforcing rules or auto-registering classes (rare, but asked in interviews).
### Answer: 
TBD

## Q20. Explain context managers with a custom example.
### Answer:
Context managers manage setup/teardown automatically using __enter__ and __exit__. Automatically closes file even if an error occurs.

In [11]:
class CustomFile:
    def __init__(self, filename):
        self.file = open(filename, 'w')

    def __enter__(self):
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.file.close()

with CustomFile('out.txt') as f:
    f.write("Hello!")

## Q21. What are dataclasses vs namedtuples vs pydantic models?
| Feature       | `dataclass`            | `namedtuple`        | `pydantic`                     |
| ------------- | ---------------------- | ------------------- | ------------------------------ |
| Mutability    | ✅ Mutable              | ❌ Immutable         | ✅ Mutable                      |
| Type Checking | Optional               | No                  | Strict                         |
| Validation    | No                     | No                  | ✅ Yes                          |
| Use Case      | Simple data containers | Lightweight structs | Config & validation-heavy apps |

## Q22. How do you optimize Python performance?
### Answer:
- Use built-ins (sum(), min(), etc.)
- Use list comprehensions over loops
- Use NumPy / Pandas for numeric tasks
- Use multiprocessing for CPU-heavy jobs
- Use asyncio / aiohttp for many I/O tasks
- Profile with cProfile or timeit

In [13]:
import timeit
print(timeit.timeit("[x*x for x in range(1000)]", number=100))

0.005385999999816704


## Q23. What are memory leaks and how to detect them?
### Answer:
Memory leaks occur when objects remain referenced unintentionally.

Causes:
- Circular references
- Global variables holding data
- Large caches not cleared

Fixes/Tools:
- gc.collect() (manual garbage collection)
- objgraph or tracemalloc for debugging

In [14]:
import tracemalloc

tracemalloc.start()

# run some code
print(tracemalloc.get_traced_memory())

(869, 10702)


## Q24. Explain shallow copy vs deep copy in nested objects.
### Answer:
(Reviewed earlier but often re-asked in advanced rounds)

In [15]:
import copy
a = [[1, 2], [3, 4]]
b = copy.copy(a)
c = copy.deepcopy(a)
a[0][0] = 99
print(b)  # [[99, 2], [3, 4]] affected
print(c)  # [[1, 2], [3, 4]] unaffected

[[99, 2], [3, 4]]
[[1, 2], [3, 4]]


## Q25. How does Python handle memory management and garbage collection?
### Answer:
- Uses reference counting and cycle detection.
- When an object’s refcount hits 0 → deallocated.
- Cyclic garbage collector handles objects referring to each other.

In [16]:
import sys
a = []
b = [a]
a.append(b)
print(sys.getrefcount(a))  # shows reference count

3


## Q26. How do you manage virtual environments in Python?
### Answer:
Isolate dependencies with venv or virtualenv.

python3 -m venv env

#### mac/linux
source env/bin/activate    
#### windows
env\Scripts\activate       

## Q27. What’s the difference between staticmethod, classmethod, and instance method?
| Method   | Decorator       | Access to `self` | Access to `cls` |
| -------- | --------------- | ---------------- | --------------- |
| Instance | None            | ✅                | ❌               |
| Class    | `@classmethod`  | ❌                | ✅               |
| Static   | `@staticmethod` | ❌                | ❌               |

In [24]:
class Example:
    def instance_m(self): 
        print("instance")
    @classmethod
    def class_m(cls): 
        print("class")
    @staticmethod
    def static_m(): 
        print("static")

Example.class_m()
Example.static_m()

class
static


## Q28. What are Python descriptors?
### Answer:
Descriptors customize attribute access using __get__, __set__, __delete__.

In [25]:
class Celsius:
    def __get__(self, instance, owner):
        return instance._temp
    def __set__(self, instance, value):
        if value < -273:
            raise ValueError("Below absolute zero!")
        instance._temp = value

class Temperature:
    temp = Celsius()

t = Temperature()
t.temp = 25
print(t.temp)

25


## Q29. Explain the use of __slots__ dunder method.
### Answer:
Reduces memory by preventing creation of __dict__.

In [26]:
class Point:
    __slots__ = ['x', 'y']
    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(1, 2)
# p.z = 3  # ❌ AttributeError

## Q30. How do you serialize and deserialize Python objects?
### Answer:
- Pickle for Python-only serialization
- JSON for cross-language compatibility

In [29]:
import pickle, json

data = {'a': 1, 'b': 2}

# Pickle
s = pickle.dumps(data)
data2 = pickle.loads(s)
print(data2)

# JSON
j = json.dumps(data)
data3 = json.loads(j)
print(data3)

{'a': 1, 'b': 2}
{'a': 1, 'b': 2}


## Bonus Topics
| Topic                    | Key Notes                             |
| ------------------------ | ------------------------------------- |
| 🧵 Thread-safe data      | Use `threading.Lock()`                |
| 🪄 Pattern matching      | `match/case` (Python 3.10+)           |
| 🧠 LRU cache             | `functools.lru_cache` for memoization |
| 🧰 Dependency management | Use `pip-tools` or `poetry`           |
| 🧩 Type checking         | `mypy --strict`                       |
| 🧍 Profiling             | `cProfile`, `line_profiler`           |
| 🧨 Debugging             | `pdb`, `ipdb`, `breakpoint()`         |